In [0]:
%pip install geopandas shapely pyproj
dbutils.library.restartPython()

In [0]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from pyspark.sql.functions import col

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:

relevant_stop_types = ["BCT", "BCS", "RLY", "RSE", "FER"]

df_stops = (
    spark.table("scottish_housing_ws.bronze.transport_stops")
    .filter(col("status") == "active")
    .filter(col("stoptype").isin(relevant_stop_types))
    .filter(col("easting").isNotNull() & col("northing").isNotNull())
    .select("atcocode", "commonname", "stoptype", "easting", "northing")
    .toPandas()
)

print(f"Active relevant stops: {len(df_stops)}")
print(df_stops["stoptype"].value_counts())

In [0]:
df_postcodes = (
    spark.table("scottish_housing_ws.silver.postcode_lookup")
    .select("postcode", "council_area_code", "easting", "northing")
    .filter(col("easting").isNotNull() & col("northing").isNotNull())
    .toPandas()
)

print(f"Postcodes loaded: {len(df_postcodes)}")

In [0]:
gdf_bus = gpd.GeoDataFrame(
    df_stops[df_stops["stoptype"].isin(["BCT", "BCS"])].copy(),
    geometry=gpd.points_from_xy(
        df_stops[df_stops["stoptype"].isin(["BCT", "BCS"])]["easting"],
        df_stops[df_stops["stoptype"].isin(["BCT", "BCS"])]["northing"]
    ),
    crs="EPSG:27700"
)

gdf_rail_ferry = gpd.GeoDataFrame(
    df_stops[df_stops["stoptype"].isin(["RLY", "RSE", "FER"])].copy(),
    geometry=gpd.points_from_xy(
        df_stops[df_stops["stoptype"].isin(["RLY", "RSE", "FER"])]["easting"],
        df_stops[df_stops["stoptype"].isin(["RLY", "RSE", "FER"])]["northing"]
    ),
    crs="EPSG:27700"
)

gdf_postcodes = gpd.GeoDataFrame(
    df_postcodes,
    geometry=gpd.points_from_xy(df_postcodes["easting"], df_postcodes["northing"]),
    crs="EPSG:27700"
)

print(f"Bus stops: {len(gdf_bus)}")
print(f"Rail/ferry stops: {len(gdf_rail_ferry)}")
print(f"Postcodes: {len(gdf_postcodes)}")

In [0]:
# Count bus stops within 500m of each postcode
# buffer each postcode point by 500m to create a circle,
# then spatial join to count how many bus stops fall inside.
# It will take a few minutes -- 161k postcodes x ~45k bus stops.

gdf_postcodes_buffered = gdf_postcodes.copy()
gdf_postcodes_buffered["geometry"] = gdf_postcodes_buffered.buffer(500)

bus_join = gpd.sjoin(
    gdf_bus[["atcocode", "geometry"]],
    gdf_postcodes_buffered[["postcode", "geometry"]],
    how="inner",
    predicate="within"
)

bus_counts = (
    bus_join.groupby("postcode")["atcocode"]
    .count()
    .reset_index()
    .rename(columns={"atcocode": "bus_stops_500m"})
)

print(f"Postcodes with at least one bus stop within 500m: {len(bus_counts)}")

In [0]:
# Count rail/ferry stops within 1000m of each postcode
# use 1000m for rail/ferry as people will walk further to a station than to a bus stop.

gdf_postcodes_buffered_1k = gdf_postcodes.copy()
gdf_postcodes_buffered_1k["geometry"] = gdf_postcodes_buffered_1k.buffer(1000)

rail_join = gpd.sjoin(
    gdf_rail_ferry[["atcocode", "geometry"]],
    gdf_postcodes_buffered_1k[["postcode", "geometry"]],
    how="inner",
    predicate="within"
)

rail_counts = (
    rail_join.groupby("postcode")["atcocode"]
    .count()
    .reset_index()
    .rename(columns={"atcocode": "rail_ferry_stops_1000m"})
)

print(f"Postcodes with rail/ferry within 1000m: {len(rail_counts)}")

In [0]:
df_silver = (
    df_postcodes[["postcode", "council_area_code"]]
    .merge(bus_counts, on="postcode", how="left")
    .merge(rail_counts, on="postcode", how="left")
    .fillna(0)
)

df_silver["bus_stops_500m"] = df_silver["bus_stops_500m"].astype(int)
df_silver["rail_ferry_stops_1000m"] = df_silver["rail_ferry_stops_1000m"].astype(int)

print(df_silver.shape)
print(df_silver.describe())

In [0]:
df_silver[df_silver["council_area_code"] == "S12000036"].head(20)

In [0]:
df_silver_spark = spark.createDataFrame(df_silver)

(
    df_silver_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.silver.transport_accessibility")
)

In [0]:
# Transit accessibility score per Scottish postcode.
# bus_stops_500m: count of active bus/coach stops within 500m of postcode centroid.
# rail_ferry_stops_1000m: count of active rail station and ferry terminal
#   entry points within 1000m of postcode centroid.
# Postcodes with no stops in range score 0.
# Source: DfT NaPTAN dataset, filtered to Scotland (ATCOCode starting with 6).
# Spatial analysis in EPSG:27700 (British National Grid).
# Distance thresholds based on standard transport accessibility research:
#   500m = ~6 min walk to bus stop, 1000m = ~12 min walk to station.